## Notebook 06: SARIMA Demo

### Purpose
Demonstrate SARIMA on 3 representative series.
Document why SARIMA cannot scale to 4000+ items.
This justifies our choice of LightGBM global model.

### What we do
1. Pick 3 series: stable, seasonal, intermittent
2. Run auto_arima on each, record time taken
3. Compute SMAPE on each
4. Calculate: if 1 series takes X minutes,
   how long for all 34,672 store-item pairs?

In [1]:
# Load and pick 3 series:

import pandas as pd
import numpy as np
import time
from pmdarima import auto_arima

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.array(y_pred, dtype=np.float32)
    return 100 * np.mean(
        2 * np.abs(y_true - y_pred) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    )

df = pd.read_parquet("../data/features/model_ready.parquet")

# Series 1: highest volume item in store 1 (stable)
top_item = (
    df[df["store_nbr"]==1]
    .groupby("item_nbr")["unit_sales"]
    .sum()
    .idxmax()
)

# Series 2: item with clear weekly pattern
# pick item with highest rolling_std_7 / rolling_mean_7 ratio
df_s1 = df[df["store_nbr"]==1].copy()
df_s1["cv"] = df_s1["rolling_std_7"] / (df_s1["rolling_mean_7"] + 1e-8)
seasonal_item = df_s1.groupby("item_nbr")["cv"].mean().idxmax()

# Series 3: most intermittent (highest zero ratio)
zero_ratio = (
    df[df["store_nbr"]==1]
    .groupby("item_nbr")["unit_sales"]
    .apply(lambda x: (x==0).mean())
)
intermittent_item = zero_ratio.idxmax()

print(f"Series 1 (stable):       item {top_item}")
print(f"Series 2 (seasonal):     item {seasonal_item}")
print(f"Series 3 (intermittent): item {intermittent_item}")
print(f"\nTotal store-item pairs in dataset: "
      f"{df.groupby(['store_nbr','item_nbr']).ngroups:,}")

Series 1 (stable):       item 1503844
Series 2 (seasonal):     item 1903500
Series 3 (intermittent): item 354971

Total store-item pairs in dataset: 34,656


In [2]:
# SARIMA on Series 1 (Stable)

# Extract series 1 — stable high volume item
s1 = (
    df[(df["store_nbr"]==1) & (df["item_nbr"]==top_item)]
    .set_index("date")["unit_sales"]
    .sort_index()
)

# Split: train on all but last 28 days
s1_train = s1.iloc[:-28]
s1_test  = s1.iloc[-28:]

print(f"Series 1 — Item {top_item}")
print(f"Train length: {len(s1_train)} days")
print(f"Test length:  {len(s1_test)} days")
print(f"Mean sales:   {s1_train.mean():.1f} units")
print(f"\nRunning auto_arima... (this may take 2-5 minutes)")

start = time.time()
model_s1 = auto_arima(
    s1_train,
    seasonal=True,
    m=7,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore"
)
time_s1 = time.time() - start

preds_s1 = model_s1.predict(n_periods=28)
preds_s1 = np.clip(preds_s1, 0, None)  # no negative predictions

smape_s1 = smape(s1_test.values, preds_s1)

print(f"\nDone.")
print(f"Time taken:  {time_s1:.1f} seconds")
print(f"Best order:  {model_s1.order}")
print(f"SMAPE:       {smape_s1:.2f}%")

Series 1 — Item 1503844
Train length: 987 days
Test length:  28 days
Mean sales:   148.9 units

Running auto_arima... (this may take 2-5 minutes)

Done.
Time taken:  53.4 seconds
Best order:  (1, 0, 0)
SMAPE:       15.54%


c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [3]:
# SARIMA on Series 2 (Seasonal)

s2 = (
    df[(df["store_nbr"]==1) & (df["item_nbr"]==seasonal_item)]
    .set_index("date")["unit_sales"]
    .sort_index()
)

s2_train = s2.iloc[:-28]
s2_test  = s2.iloc[-28:]

print(f"Series 2 — Item {seasonal_item}")
print(f"Train length: {len(s2_train)} days")
print(f"Mean sales:   {s2_train.mean():.1f} units")
print(f"\nRunning auto_arima...")

start = time.time()
model_s2 = auto_arima(
    s2_train,
    seasonal=True,
    m=7,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore"
)
time_s2 = time.time() - start

preds_s2 = model_s2.predict(n_periods=28)
preds_s2 = np.clip(preds_s2, 0, None)

smape_s2 = smape(s2_test.values, preds_s2)

print(f"\nDone.")
print(f"Time taken:  {time_s2:.1f} seconds")
print(f"Best order:  {model_s2.order}")
print(f"SMAPE:       {smape_s2:.2f}%")

Series 2 — Item 1903500
Train length: 566 days
Mean sales:   14.6 units

Running auto_arima...

Done.
Time taken:  16.5 seconds
Best order:  (4, 0, 0)
SMAPE:       114.81%


c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [6]:
# SARIMA on Series 3 (Intermittent)
# Check what zero percentages actually exist in store 1
print("Store 1 zero% distribution:")
print(store1_items["zero_pct"].describe())
print(f"\nItems with zero_pct >= 0.1: "
      f"{(store1_items['zero_pct'] >= 0.1).sum()}")
print(f"Items with zero_pct >= 0.05: "
      f"{(store1_items['zero_pct'] >= 0.05).sum()}")
print(f"Items with total_rows >= 100: "
      f"{(store1_items['total_rows'] >= 100).sum()}")

# Use relaxed criteria
intermittent_candidates = store1_items[
    (store1_items["total_rows"] >= 100) &
    (store1_items["zero_pct"] >= 0.05)
].sort_values("zero_pct", ascending=False)

print(f"\nCandidates found: {len(intermittent_candidates)}")

if len(intermittent_candidates) == 0:
    # Just pick item with highest zero% regardless of threshold
    intermittent_candidates = store1_items[
        store1_items["total_rows"] >= 100
    ].sort_values("zero_pct", ascending=False)
    print("Using relaxed criteria — highest zero% with 100+ rows")

intermittent_item_s1 = int(
    intermittent_candidates.iloc[0]["item_nbr"]
)
print(f"\nSelected item: {intermittent_item_s1}")
print(f"Zero %: "
      f"{intermittent_candidates.iloc[0]['zero_pct']*100:.1f}%")
print(f"Rows:   "
      f"{intermittent_candidates.iloc[0]['total_rows']:.0f}")

# Extract series
s3 = (
    df[(df["store_nbr"]==1) &
       (df["item_nbr"]==intermittent_item_s1)]
    .set_index("date")["unit_sales"]
    .sort_index()
)

s3_train = s3.iloc[:-28]
s3_test  = s3.iloc[-28:]

print(f"\nTrain length: {len(s3_train)} days")
print(f"Mean sales:   {s3_train.mean():.1f} units")
print(f"Zero days:    {(s3_train==0).sum()} "
      f"({(s3_train==0).mean()*100:.1f}%)")
print(f"\nRunning auto_arima...")

start = time.time()
model_s3 = auto_arima(
    s3_train,
    seasonal=True,
    m=7,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore"
)
time_s3 = time.time() - start

preds_s3 = model_s3.predict(n_periods=28)
preds_s3 = np.clip(preds_s3, 0, None)
smape_s3 = smape(s3_test.values, preds_s3)

print(f"\nDone.")
print(f"Time taken: {time_s3:.1f} seconds")
print(f"Order:      {model_s3.order}")
print(f"SMAPE:      {smape_s3:.2f}%")

Store 1 zero% distribution:
count    3565.000000
mean        0.000518
std         0.012607
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.500000
Name: zero_pct, dtype: float64

Items with zero_pct >= 0.1: 3
Items with zero_pct >= 0.05: 3
Items with total_rows >= 100: 3176

Candidates found: 0
Using relaxed criteria — highest zero% with 100+ rows

Selected item: 1230244
Zero %: 3.4%
Rows:   119

Train length: 91 days
Mean sales:   1.1 units
Zero days:    2 (2.2%)

Running auto_arima...

Done.
Time taken: 1.4 seconds
Order:      (0, 0, 0)
SMAPE:      32.90%


c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\innso\anaconda3\envs\forecast\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [7]:
# The Key Conclusion

# This cell produces the most important insight from SARIMA
# It justifies why we use LightGBM global model instead

total_time_3_series = time_s1 + time_s2 + time_s3
avg_time_per_series = total_time_3_series / 3
total_pairs = 34656

estimated_total_minutes = (
    avg_time_per_series * total_pairs / 60
)
estimated_total_hours = estimated_total_minutes / 60

print("=" * 50)
print("SARIMA SCALING ANALYSIS")
print("=" * 50)
print(f"Series 1 (stable):       {time_s1:.1f}s  "
      f"SMAPE={smape_s1:.1f}%")
print(f"Series 2 (seasonal):     {time_s2:.1f}s  "
      f"SMAPE={smape_s2:.1f}%")
print(f"Series 3 (intermittent): {time_s3:.1f}s  "
      f"SMAPE={smape_s3:.1f}%")
print(f"\nAverage time per series: {avg_time_per_series:.1f}s")
print(f"Total store-item pairs:  {total_pairs:,}")
print(f"\nEstimated time for ALL series:")
print(f"  {estimated_total_minutes:,.0f} minutes")
print(f"  {estimated_total_hours:,.0f} hours")
print(f"\nConclusion:")
print(f"SARIMA would take ~{estimated_total_hours:.0f} hours")
print(f"to fit all {total_pairs:,} store-item pairs.")
print(f"This is not viable for production forecasting.")
print(f"\nThis is why we use LightGBM global model:")
print(f"One model, trained once, predicts all series.")
print("=" * 50)

SARIMA SCALING ANALYSIS
Series 1 (stable):       53.4s  SMAPE=15.5%
Series 2 (seasonal):     16.5s  SMAPE=114.8%
Series 3 (intermittent): 1.4s  SMAPE=32.9%

Average time per series: 23.8s
Total store-item pairs:  34,656

Estimated time for ALL series:
  13,735 minutes
  229 hours

Conclusion:
SARIMA would take ~229 hours
to fit all 34,656 store-item pairs.
This is not viable for production forecasting.

This is why we use LightGBM global model:
One model, trained once, predicts all series.
